# SCConv 3 lớp — tối đa 70 epoch — dataset_v116

Bản dữ liệu được chấp nhận hiện có **572/2.000 ảnh**; train 432, validation 72, test 68. Đây là tiến độ số lượng, không phải độ chính xác model. Ảnh pending không được đóng gói để train. Mục tiêu 2.000 ảnh chưa hoàn thành.

Chọn Runtime → Change runtime type → GPU. Tải `training_colab_70_v116.zip` vào `MyDrive/HelmetSCConv70`. Đọc README.md và giấy phép từng nguồn, đặc biệt nguồn CC BY-NC-SA 4.0 chỉ dùng phi thương mại.

Chạy SCConv trước, baseline là lựa chọn đối chứng sau. 70 là tổng số epoch kể cả resume, patience20, imgsz768, batch thử8→4→2. Colab không bảo đảm hoàn tất một phiên. Không thay checkpoint demo.


In [ ]:
import sys, subprocess
subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'ultralytics==8.4.155'])
# Giữ PyTorch CUDA do Colab cung cấp; kiểm chứng runtime/preflight ở các ô sau.


In [ ]:
from google.colab import drive
from pathlib import Path
import os, json, zipfile, hashlib
drive.mount('/content/drive')
DRIVE = Path('/content/drive/MyDrive/HelmetSCConv70')
ARCHIVE = DRIVE / 'training_colab_70_v116.zip'
LOCAL = Path('/content/helmet_scconv70')
LOCAL.mkdir(parents=True, exist_ok=True)
assert ARCHIVE.is_file(), f'Hãy tải training_colab_70_v116.zip vào {DRIVE}'
with zipfile.ZipFile(ARCHIVE) as archive:
    for entry in archive.infolist():
        target = (LOCAL / entry.filename).resolve()
        assert target.is_relative_to(LOCAL.resolve()), 'Đường dẫn ZIP không hợp lệ'
    archive.extractall(LOCAL)
for name, expected in json.loads((LOCAL/'files_sha256.json').read_text()).items():
    assert hashlib.sha256((LOCAL/name).read_bytes()).hexdigest() == expected, name
CODE = LOCAL / 'code'
DATA = LOCAL / 'dataset_v116'
os.chdir(LOCAL)
sys.path.insert(0, str(CODE))
import torch, ultralytics
from train_colab import verify_dataset, preflight_batch, run, validate_resume_checkpoint, PRETRAINED_SHA256
assert ultralytics.__version__ == '8.4.155'
assert torch.cuda.is_available(), 'Chọn runtime GPU trước khi chạy chính'
print('Python:', sys.version)
print('Torch:', torch.__version__, 'CUDA:', torch.version.cuda, 'GPU:', torch.cuda.get_device_name(0))
verify_dataset(DATA/'data.yaml')
print(json.loads((DATA/'statistics.json').read_text()))


## Kiểm tra train/save/resume trước run chính
Chỉ vài ảnh train/val, 3 epoch tổng, kích thước 64 để kiểm tra kỹ thuật; đồng thời kiểm tra head forward 768. Kết quả không dùng đánh giá chất lượng. Snapshot smoke được ghi vào Drive.


In [ ]:
from datetime import datetime
SMOKE_NAME = 'smoke_' + datetime.now().strftime('%Y%m%d_%H%M%S')
subprocess.check_call([sys.executable, str(CODE/'smoke_test.py'), '--data', str(DATA),
                       '--output', str(LOCAL/SMOKE_NAME), '--backup-dir', str(DRIVE/SMOKE_NAME)])
assert json.loads((LOCAL/SMOKE_NAME/'result.json').read_text())['passed']


## Cấu hình run
Lần đầu `RESUME=False`. Sau khi bị ngắt: giữ nguyên tên run/dataset, chọn `RESUME=True`. Snapshot `resume.pt` khôi phục optimizer/LR/epoch. `resume.previous.pt` dùng khi snapshot chính hỏng. Đổi tên run khi thử baseline; không chạy hai mô hình cùng lúc.


In [ ]:
VARIANT = 'scconv'  # 'baseline' để chạy đối chứng sau, dùng tên run khác
RUN_NAME = 'scconv_clean70_v116'
RESUME = False
RUN_DIR = LOCAL/'runs'/RUN_NAME
BACKUP_DIR = DRIVE/RUN_NAME
RESUME_FILE = BACKUP_DIR/'resume.pt'  # hoặc 'resume.previous.pt'
PRETRAINED = LOCAL/'pretrained/yolo26s.pt'
assert hashlib.sha256(PRETRAINED.read_bytes()).hexdigest() == PRETRAINED_SHA256
if RESUME:
    payload = torch.load(RESUME_FILE, map_location='cpu', weights_only=False)
    validate_resume_checkpoint(payload, 70)
    assert payload['helmet_training']['variant'] == VARIANT
    assert payload['train_args']['imgsz'] == 768
    BATCH = int(payload['train_args']['batch'])
    print('Tiếp tục từ epoch', payload['epoch']+2, 'đến tối đa 70')
    del payload
    preflight_batch(str(PRETRAINED), VARIANT, candidates=(BATCH,))
else:
    BATCH = preflight_batch(str(PRETRAINED), VARIANT)
print('Batch đã chốt:', BATCH, '| imgsz=768 | epochs=70 | patience=20')


In [ ]:
trainer = run(DATA/'data.yaml', RUN_DIR, BACKUP_DIR, str(PRETRAINED),
              variant=VARIANT, resume=RESUME_FILE if RESUME else None, batch=BATCH)
print('Checkpoint và CSV trên Drive:', BACKUP_DIR)
print('Không dùng checkpoint smoke để demo.')


## Validation và chọn ngưỡng
Ưu tiên checkpoint có AP50–95 lớp 2 tốt nhất. `best.pt` vẫn giữ best theo fitness tổng. AP, P/R, ảnh lỗi và ngưỡng F1 lớp 2 chỉ được tính từ validation. Xem ảnh lỗi trước khi quyết định tích hợp; confidence cao không tự chứng minh dự đoán đúng.


In [ ]:
BEST = BACKUP_DIR/'best-class2.pt'
assert BEST.exists(), 'Chưa có checkpoint lớp 2 hợp lệ'
EVALUATION_NAME = 'val_' + datetime.now().strftime('%Y%m%d_%H%M%S')
VAL_OUTPUT = DRIVE/RUN_NAME/EVALUATION_NAME
subprocess.check_call([sys.executable, str(CODE/'evaluate_model.py'), '--model', str(BEST),
                       '--data', str(DATA/'data.yaml'), '--output', str(VAL_OUTPUT)])
print((VAL_OUTPUT/'metrics.json').read_text())
print('Đề xuất ngưỡng:', VAL_OUTPUT/'selection.json')


## Test cuối — chỉ sau khi khóa mô hình và ngưỡng
Test có 68 ảnh và 9 box lớp không đội mũ. Số mẫu ít và phụ thuộc nguồn; không suy rộng kết quả sang mọi camera. Giữ test khóa, chỉ chọn ngưỡng/epoch từ validation. Ô test tắt mặc định; marker chống chạy lặp vẫn được giữ.


In [ ]:
RUN_FINAL_TEST = False
if RUN_FINAL_TEST:
    TEST_OUTPUT = DRIVE/RUN_NAME/'final_test'
    subprocess.check_call([sys.executable, str(CODE/'evaluate_model.py'), '--model', str(BEST),
                           '--data', str(DATA/'data.yaml'), '--output', str(TEST_OUTPUT),
                           '--final-test', '--selection', str(VAL_OUTPUT/'selection.json')])
else:
    print('Chưa chạy test cuối. Giữ test khóa; chỉ đánh giá sau khi chốt mô hình và ngưỡng.')
